# Automated label transfer with Cell2location

This script runs Cell2location to transfer labels from a single cell dataset to a spatial dataset of the same tissue

Kleshchevnikov, V., Shmatko, A., Dann, E. et al. Cell2location maps fine-grained cell types in spatial transcriptomics. Nat Biotechnol (2022). https://doi.org/10.1038/s41587-021-01139-4 https://www.nature.com/articles/s41587-021-01139-4

In [ ]:
# Packages
import scanpy as sc
#import squidpy as sq
import numpy as np
import seaborn as sns
import pandas as pd
import cell2location as c2l
import matplotlib
import os
import scipy
import scipy.sparse
import matplotlib.pyplot as plt

from matplotlib import rcParams
from cycler import cycler
from cell2location.utils.filtering import filter_genes

# Settings
sc.settings.set_figure_params(dpi=80, facecolor="white")
rcParams['pdf.fonttype'] = 42
rcParams['figure.figsize'] = 6,6

In [ ]:
# Set up results directory
os.mkdir = ''
os.mkdir = ''

In [ ]:
# Set dir paths
results = ''
fig_dir = ''
data_dir = ''
ref_dir = ''

In [ ]:
# Creating paths and names to results folders for reference regression and cell2location models
pt1_run_name = f'{results}/Reference_signatures' # This only needs to be done once, unless using a new reference
pt2_run_name = f'{results}/Spatial_mapping'

## Cell2location label transfer

### (1) Generate gene signatures from atlas

Reading in the spatial and the reference data

In [ ]:
# Spatial
adata_st = sc.read_h5ad(f'{data_dir}orv01_spatial_clustered.h5ad')
adata_st

In [ ]:
# Reference atlas
adata_sc = sc.read_h5ad(f'{ref_dir}adata_object_orv01_scrnaseq.h5ad')
adata_sc

In [ ]:
adata_sc.layers['counts'] = adata_sc.X.copy()

In [ ]:
adata_sc.obs['orig.ident'].unique()

In [ ]:
# Subset reference to only d10
adata_sc = adata_sc[adata_sc.obs['orig.ident']=='D10resect']

In [ ]:
# Making sure reference data has integer count values

# Processing in chunks to avoid memory limits
def process_in_chunks(sparse_matrix, chunk_size=1000):
    # Number of rows in the matrix
    num_rows = sparse_matrix.shape[0]
    
    # Initialize a list to store the processed chunks
    processed_chunks = []
    
    for start in range(0, num_rows, chunk_size):
        end = min(start + chunk_size, num_rows)
        chunk = sparse_matrix[start:end, :].toarray()
        chunk_int = np.round(chunk).astype(int)
        processed_chunks.append(scipy.sparse.csr_matrix(chunk_int))
    
    # Concatenate all chunks back into a single sparse matrix
    return scipy.sparse.vstack(processed_chunks)

# Process the sparse matrix in chunks
sparse_matrix_int = process_in_chunks(adata_sc.X, chunk_size=1000)

# Update the AnnData object with the processed sparse matrix
adata_sc.X = sparse_matrix_int

In [ ]:
# Match genes between spatial and reference
adata_st.vars

In [ ]:
# Filter reference data
fig = plt.gcf()
selected = filter_genes(adata_sc, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12) # default, recommended as starting point
# Filter the object based on selected genes
adata_ref_fil = adata_sc[:, selected].copy()
plt.savefig(f"{fig_dir}c2l_gene_filter_plot.png", dpi=400, bbox_inches='tight')

In [ ]:
# Save filtered reference
adata_ref_fil.write(f'Reference_c2l_filtered.h5ad') # Insert full file path

Model set up -- the next cell is ran in a separate python script that reads in the adata_ref_fil object we just saved

In [ ]:
# Prepare anndata for the regression model
c2l.models.RegressionModel.setup_anndata(adata=adata_ref_fil,
                        # batch_key='Source', # Only a single sample in reference
                        # cell type, covariate used for constructing signatures
                        labels_key='cell_type_final',
                        # no covariates to add
                       )

# Create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref_fil)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

# Train the model..... python script ran through terminal

Now checking the results:

In [ ]:
# Read in model and results
adata_file = f"{pt1_run_name}/reference_signatures.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = c2l.models.RegressionModel.load(f"{pt1_run_name}", adata_ref)

In [ ]:
# Plotting model history
mod.plot_history(20)
plt.savefig(f'{fig_dir}c2l_pt1_elbo_plot.png', dpi=400)

In [ ]:
adata_ref

In [ ]:
if not hasattr(mod, 'samples'):
    # Reload the posterior samples from the AnnData object if available
    mod.samples = adata_ref.uns['mod']  # Adjust this based on how you saved the posterior

In [ ]:
mod.plot_QC()
plt.savefig(f"{fig_dir}c2l_part1_reconstruction_plot.png", dpi=400, bbox_inches='tight')
plt.close()

In [ ]:
adata_ref.uns['mod']['factor_names']

In [ ]:
# Extracting reference cell types signatures as a pd.DataFrame (inf_aver)

# For spatial mapping we just need the estimated expression of every gene in every cell type

# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5]

### (2) Map gene signatures to spatial data

Spatial mapping is ran as a separate python script through terminal

In [ ]:
adata_st

In [ ]:
# Removed MT genes in spatial data
adata_st.var["MT_gene"] = [gene.startswith("MT-") for gene in adata_st.var_names]

# Remove genes
adata_st.obsm["MT-"] = adata_st[:, adata_st.var["MT_gene"].values].X.toarray()
adata_st = adata_st[:, ~adata_st.var["MT_gene"].values]

In [ ]:
adata_st.obs['sample'] = list(adata_st.uns['spatial'].keys())[0]

In [ ]:
# Find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_st.var_names, inf_aver.index)
adata_st = adata_st[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

In [ ]:
inf_aver

In [ ]:
# Set layer of log counts
# adata_st.layers['log_counts'] = adata_st.X

# Bring raw counts to top
adata_st.X = adata_st.layers['counts']

In [ ]:
# Prepare anndata for cell2location model
c2l.models.Cell2location.setup_anndata(adata=adata_st) #, batch_key="tissue") -- no batch key 

In [ ]:
# Create model
mod = c2l.models.Cell2location(
    adata_st, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=8, # 8 = estimated cells per spot for standard (spot-based) Visium data
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)

In [ ]:
print('Training the model....')
mod.train(max_epochs=15000, # Testing first at 15000 
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
          accelerator='gpu'
         )

In [ ]:
# Python script runs model training

Checking cell label predictions

In [ ]:
# Reload in model and adata object
adata_file = f"{pt2_run_name}/c2l_label_predications.h5ad"
adata_st = sc.read_h5ad(adata_file)

# Read in model
mod = c2l.models.Cell2location.load(f"{pt2_run_name}", adata_st)

In [ ]:
# Plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training'])
plt.tight_layout()
plt.savefig(f'{fig_dir}c2l_pt2_elbo_plot.png', dpi=400)

In [ ]:
if not hasattr(mod, 'samples'):
    # Reload the posterior samples from the AnnData object if available
    mod.samples = adata_st.uns['mod']  # Adjust this based on how you saved the posterior

In [ ]:
import matplotlib.pyplot as plt
mod.plot_QC()
plt.savefig(f'{fig_dir}c2l_part2_reconstruction_plot.png', dpi=400)

In [ ]:
# Extract predictions
adata_st.obs[adata_st.uns['mod']['factor_names']] = adata_st.obsm['q05_cell_abundance_w_sf']

In [ ]:
import importlib
import os
importlib.reload(os)

Plotting cell label predictions spatially

In [ ]:
from pathlib import Path

sc.settings.figdir = fig_dir   # or an absolute path, e.g. "/path/to/output"
sc.settings.set_figure_params(dpi_save=400)
sc.settings.plot_prefix = ""

# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting

# select one slide
from cell2location.utils import select_slide
slide = select_slide(adata_st, 'orv01')

# plot in spatial coordinates
with plt.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(slide, cmap='magma',
                  # show first 8 cell types
                  color=['+Prolif B cells', '+Ribo/-Prolif B cells', '-Ribo B cells',
                         'CD4+ T cells', 'CD8+ T cells', 'IL23R+ T cells', 'Immune cells',
                         'NK cells', 'Naive B cells', 'Naive T cells', 'Proliferating T cells'],
                  ncols=4, size=1.3,
                  img_key='hires',
                  # limit color scale at 99.2% quantile of cell abundance
                  vmin=0, vmax='p99.2',save='c2l_spatial_cell_type_plots.png')

In [ ]:
# Plot just the cell types we want
with plt.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 4.5]}):

    sc.pl.spatial(slide, cmap='magma',
                  # show first 8 cell types
                  color=['+Prolif B cells', '+Ribo/-Prolif B cells', '-Ribo B cells',
                         'CD4+ T cells', 'CD8+ T cells', 'NK cells', 'Proliferating T cells', 'IL23R+ T cells', 'Myeloid cells'],
                  ncols=3, size=1.3,
                  img_key='hires',
                  # limit color scale at 99.2% quantile of cell abundance
                  vmin=0, vmax='p99.2',save='c2l_spatial_cell_type_plots_paper_fig.png')

In [ ]:
# select up to 6 clusters
bcell_clust_labels = ['+Prolif B cells', '+Ribo/-Prolif B cells', '-Ribo B cells']
bcell_clust_col = ['' + str(i) for i in bcell_clust_labels] # in case column names differ from labels

tcell_clust_labels = ['CD4+ T cells', 'CD8+ T cells', 'IL23R+ T cells', 'Myeloid cells', 'Proliferating T cells', 'NK cells']
tcell_clust_col = ['' + str(i) for i in tcell_clust_labels] # in case column names differ from labels


slide = select_slide(adata_st, 'orv01')

# Bcells
with matplotlib.rc_context({"figure.figsize": (15, 15)}):
    fig = c2l.plt.plot_spatial(
        adata=slide,
        color=bcell_clust_col,
        labels=bcell_clust_labels,
        max_color_quantile=0.992, # sets max at 99.2 quantile; anything above is shown to be most saturated
        circle_diameter=6,
        show_img=True,
        colorbar_position="right",
        colorbar_shape={"horizontal_gaps": 0.2}
    )
    plt.savefig(f"{fig_dir}/bcells_spatial_plot.png", dpi=400, bbox_inches="tight")
    plt.close()



# Tcells
with matplotlib.rc_context({"figure.figsize": (15, 15)}):
    fig = c2l.plt.plot_spatial(
        adata=slide,
        color=tcell_clust_col,
        labels=tcell_clust_labels,
        max_color_quantile=0.992, # sets max at 99.2 quantile; anything above is shown to be most saturated
        circle_diameter=6,
        show_img=True,
        colorbar_position="right",
        colorbar_shape={"horizontal_gaps": 0.2}
    )
    plt.savefig(f"{fig_dir}/tcells_spatial_plot.png", dpi=400, bbox_inches="tight")
    plt.close()

In [ ]:
cell_type_columns = adata_st.uns['mod']['factor_names']
cell_abundance_df = adata_st.obs[cell_type_columns]

# Step 2: Find the cell type with the highest abundance for each spot in data
# This will give you a Series where each value is the name of the cell type with the highest quantile for that cell
most_appropriate_cell_type = cell_abundance_df.idxmax(axis=1)

adata_st.obs['c2l_predicted'] = most_appropriate_cell_type

In [ ]:
### We don't want to do this, because it assigns a single cell type to each spot -- when we know each spot likely contains multiple
### You can see that some cell types are missing (i.e. CD4+ T cells)
### But let's continue to see what it looks like
adata_st.obs["c2l_predicted"].value_counts()

In [ ]:
adata_st.obs

In [ ]:
adata_st.obs['c2l_predicted'].unique()

In [ ]:
# Group into broad biological categories and save to adata.obs

# Fine groupings
grouping_rules = {
    'T cells': ['CD8+ T cells', 'CD4+ T cells', 'Naive T cells', 'IL23R+ T cells',
               'CD4+ T cells', 'NK cells', 'Proliferating T cells'],
    'B cells': ['+Ribo/-Prolif B cells', '-Ribo B cells', '+Prolif B cells'],
    'Myeloid cells': ['Myeloid cells'],
}

# Function to map cell types to grouped annotations based on the provided rules
def map_to_group(cell_type, rules):
    for group, types in rules.items():
        if cell_type in types:
            return group
    # Return original cell type if no group is found
    return cell_type

# Apply the function to create a new column for grouped annotations
adata_st.obs['grouped_annotations'] = adata_st.obs['c2l_predicted'].apply(lambda x: map_to_group(x, grouping_rules))

adata_st.obs

In [ ]:
adata_st.obs['grouped_annotations'].value_counts()

Plot annotated UMAP

In [ ]:
# Confirming raw counts are set
print(adata_st.X[1:100,1:100])

In [ ]:
# Checking on original UMAP coords
sc.pl.umap(adata_st, color='c2l_predicted')

In [ ]:
# Grouped annotations
sc.pl.umap(adata_st, color='grouped_annotations')

In [ ]:
# Save c2l labelled object here:
adata_st.write(f'{data_dir}orv01_spatial_c2l_annotated.h5ad')

In [ ]:
adata_st = sc.read_h5ad(f'{data_dir}orv01_spatial_c2l_annotated.h5ad')

In [ ]:
adata_st